In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

CSV_DIR = '../lighthouse_csv_v7/lighthouse_csv_v7/'

print('--- Building Recommendation Engine ---')
residents_df = pd.read_csv(os.path.join(CSV_DIR, 'residents.csv'))
incidents_df = pd.read_csv(os.path.join(CSV_DIR, 'incident_reports.csv'))

incident_counts = incidents_df.groupby('resident_id').size().reset_index(name='incident_count')
res_features = pd.merge(residents_df, incident_counts, on='resident_id', how='left')
res_features['incident_count'] = res_features['incident_count'].fillna(0)

res_features['date_of_birth'] = pd.to_datetime(res_features['date_of_birth'], errors='coerce')
res_features['age_years'] = ((pd.Timestamp.today() - res_features['date_of_birth']).dt.days / 365.25).fillna(0)

features = res_features[['age_years', 'incident_count']].fillna(0)
features_scaled = StandardScaler().fit_transform(features)
similarity_matrix = cosine_similarity(features_scaled)

recommendations = []
for idx, row in res_features.iterrows():
    similar_indices = similarity_matrix[idx].argsort()[-4:-1][::-1]
    similar_resident_ids = res_features.iloc[similar_indices]['resident_id'].values
    matches = list(similar_resident_ids) + [None, None, None]
    recommendations.append({
        'resident_id': row['resident_id'],
        'recommended_match_1': matches[0],
        'recommended_match_2': matches[1],
        'recommended_match_3': matches[2],
    })

pd.DataFrame(recommendations).to_csv('Resident_Recommendations.csv', index=False)
print("Saved 'Resident_Recommendations.csv'.")
print('')
print('--- Building Predictive Model ---')

supporters_df = pd.read_csv(os.path.join(CSV_DIR, 'supporters.csv'))
donations_df = pd.read_csv(os.path.join(CSV_DIR, 'donations.csv'))

donations_df['monetary_value'] = donations_df['amount'].fillna(donations_df['estimated_value']).fillna(0)
donor_totals = donations_df.groupby('supporter_id')['monetary_value'].sum().reset_index()
donation_counts = donations_df.groupby('supporter_id').size().reset_index(name='donation_count')

donor_data = supporters_df.merge(donor_totals, on='supporter_id', how='left').merge(donation_counts, on='supporter_id', how='left')
donor_data['monetary_value'] = donor_data['monetary_value'].fillna(0)
donor_data['donation_count'] = donor_data['donation_count'].fillna(0)

threshold = donor_data['monetary_value'].quantile(0.75)
donor_data['is_high_value'] = (donor_data['monetary_value'] >= threshold).astype(int)

X_pred = pd.get_dummies(
    donor_data[['supporter_type', 'relationship_type', 'status', 'acquisition_channel']],
    drop_first=True,
)
X_pred['donation_count'] = donor_data['donation_count']
y_pred = donor_data['is_high_value']

X_train, X_test, y_train, y_test = train_test_split(X_pred, y_pred, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

print(f'Predictive Model Accuracy: {clf.score(X_test, y_test):.2f}')
print('')
print('Confusion Matrix:')
print(confusion_matrix(y_test, predictions))
print('')
print('Classification Report:')
print(classification_report(y_test, predictions))
print('')
print('--- Building Explanatory Model ---')

social_df = pd.read_csv(os.path.join(CSV_DIR, 'social_media_posts.csv'))
social_df['caption_length'] = social_df['caption_length'].fillna(social_df['caption'].fillna('').str.len())
social_df['post_hour'] = social_df['post_hour'].fillna(12)
social_df['total_engagement'] = social_df['likes'].fillna(0) + social_df['shares'].fillna(0) + social_df['comments'].fillna(0)

X_exp = social_df[['caption_length', 'post_hour']].fillna(0)
y_exp = social_df['total_engagement'].fillna(0)

reg = LinearRegression()
reg.fit(X_exp, y_exp)

coefficients = pd.DataFrame({'feature': X_exp.columns, 'coefficient': reg.coef_})
print('')
print('--- EXPLANATORY MODEL INSIGHTS ---')
print('Intercept:', reg.intercept_)
print(coefficients.sort_values(by='coefficient', ascending=False))

plt.figure(figsize=(8, 4))
plt.bar(coefficients['feature'], coefficients['coefficient'], color='#059669')
plt.title('Drivers of Social Media Engagement')
plt.ylabel('Coefficient (Impact on Engagement)')
plt.tight_layout()
plt.savefig('Explanatory_Insights_Chart.png')
print('')
print("Saved 'Explanatory_Insights_Chart.png'.")

